In [2]:
# ============================================================
# Імпорти
# ============================================================
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Створюємо Spark-сесію (точка входу для роботи зі Spark)
spark = SparkSession.builder.appName("HW1_PySpark").getOrCreate()

# Папка, де лежить скрипт
# base_dir = os.path.dirname(os.path.abspath(__file__))
base_dir = os.getcwd()

# ============================================================
# 1. Завантаження CSV-файлів як окремих DataFrame
# ============================================================
# header=True      — перший рядок містить назви колонок
# inferSchema=True — Spark сам визначає типи (int, double, date...)
users_df = spark.read.csv(
    os.path.join(base_dir, "users.csv"), header=True, inferSchema=True
)
purchases_df = spark.read.csv(
    os.path.join(base_dir, "purchases.csv"), header=True, inferSchema=True
)
products_df = spark.read.csv(
    os.path.join(base_dir, "products.csv"), header=True, inferSchema=True
)

users_df.show(5)
purchases_df.show(5)
products_df.show(5)

+-------+------+---+-----------------+
|user_id|  name|age|            email|
+-------+------+---+-----------------+
|      1|User_1| 45|user1@example.com|
|      2|User_2| 48|user2@example.com|
|      3|User_3| 36|user3@example.com|
|      4|User_4| 46|user4@example.com|
|      5|User_5| 29|user5@example.com|
+-------+------+---+-----------------+
only showing top 5 rows
+-----------+-------+----------+----------+--------+
|purchase_id|user_id|product_id|      date|quantity|
+-----------+-------+----------+----------+--------+
|          1|     52|         9|2022-01-01|       1|
|          2|     93|        37|2022-01-02|       8|
|          3|     15|        33|2022-01-03|       1|
|          4|     72|        42|2022-01-04|       9|
|          5|     61|        44|2022-01-05|       6|
+-----------+-------+----------+----------+--------+
only showing top 5 rows
+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------

In [3]:
# ============================================================
# 2. Очищення даних: видаляємо рядки з пропущеними значеннями
# ============================================================
print("До очищення:", users_df.count(), purchases_df.count(), products_df.count())

users_df = users_df.dropna()
purchases_df = purchases_df.dropna()
products_df = products_df.dropna()

print("Після очищення:", users_df.count(), purchases_df.count(), products_df.count())

До очищення: 100 200 50
Після очищення: 95 195 47


In [4]:
# ============================================================
# 3. Загальна сума покупок за кожною категорією продуктів
# ============================================================
# З'єднуємо покупки з продуктами, щоб отримати ціну та категорію
purchases_products_df = purchases_df.join(products_df, on="product_id", how="inner")

# Сума покупки = кількість * ціна одиниці
purchases_products_df = purchases_products_df.withColumn(
    "total_price", F.col("quantity") * F.col("price")
)

total_by_category = (
    purchases_products_df.groupBy("category")
    .agg(F.round(F.sum("total_price"), 2).alias("total_spent"))
    .orderBy(F.desc("total_spent"))
)

print("Загальна сума покупок за категоріями:")
total_by_category.show()

Загальна сума покупок за категоріями:
+-----------+-----------+
|   category|total_spent|
+-----------+-----------+
|     Sports|     1802.5|
|       Home|     1523.5|
|Electronics|     1174.8|
|   Clothing|      790.3|
|     Beauty|      459.9|
+-----------+-----------+



In [5]:
# ============================================================
# 4. Сума покупок за категоріями для віку 18–25 включно
# ============================================================
# Додаємо дані користувачів, щоб знати вік
full_df = purchases_products_df.join(users_df, on="user_id", how="inner")

# between() включає обидві межі: 18 <= age <= 25
young_df = full_df.filter(F.col("age").between(18, 25))

young_by_category = young_df.groupBy("category").agg(
    F.sum("total_price").alias("total_spent")
)

print("Сума покупок за категоріями (вік 18–25):")
young_by_category.withColumn("total_spent", F.round("total_spent", 2)).orderBy(
    F.desc("total_spent")
).show()

Сума покупок за категоріями (вік 18–25):
+-----------+-----------+
|   category|total_spent|
+-----------+-----------+
|       Home|      361.1|
|     Sports|      310.5|
|Electronics|      249.6|
|   Clothing|      245.0|
|     Beauty|       41.4|
+-----------+-----------+



In [6]:
# ============================================================
# 5. Частка покупок кожної категорії від сумарних витрат (вік 18–25)
# ============================================================
# Загальна сума витрат групи 18–25 (одне число)
total_young_spent = young_df.agg(F.sum("total_price")).collect()[0][0]

# Відсоток = сума категорії / загальна сума * 100, округлення до 2 знаків
young_share = young_by_category.withColumn(
    "percentage", F.round(F.col("total_spent") / F.lit(total_young_spent) * 100, 2)
).withColumn("total_spent", F.round("total_spent", 2))

print(f"Загальні витрати (вік 18–25): {round(total_young_spent, 2)}")
print("Частка витрат за категоріями (вік 18–25), %:")
young_share.orderBy(F.desc("percentage")).show()

Загальні витрати (вік 18–25): 1207.6
Частка витрат за категоріями (вік 18–25), %:
+-----------+-----------+----------+
|   category|total_spent|percentage|
+-----------+-----------+----------+
|       Home|      361.1|      29.9|
|     Sports|      310.5|     25.71|
|Electronics|      249.6|     20.67|
|   Clothing|      245.0|     20.29|
|     Beauty|       41.4|      3.43|
+-----------+-----------+----------+



In [7]:
# ============================================================
# 6. Топ-3 категорії з найвищим відсотком витрат (вік 18–25)
# ============================================================
top3_categories = young_share.orderBy(F.desc("percentage")).limit(3)

print("Топ-3 категорії (вік 18–25):")
top3_categories.select("category", "percentage").show()


# Завершуємо сесію
spark.stop()

Топ-3 категорії (вік 18–25):
+-----------+----------+
|   category|percentage|
+-----------+----------+
|       Home|      29.9|
|     Sports|     25.71|
|Electronics|     20.67|
+-----------+----------+

